# Sourcing Desk: step-by-step walkthrough

Run the cells top to bottom. Each step is the same function the pipeline calls, and every call is also
recorded to `runs/<run_id>/<step>__<part>.json` (input, output, API calls, timing, errors).

- Crustdata responses are cached in `.cache/crustdata/`, so re-running a cell costs **0 credits**.
  Set `use_cache=False` to force fresh data.
- Kernel: pick the repo's `.venv` (Python 3.13).

In [ ]:
import json
import pandas as pd
from desk import ROOT, pipeline, steps, view
from desk.crustdata import Crustdata
from desk.trace import Run, list_runs, load_run

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)

# electronics_bom.yaml = the founder's words only (Gemini reads it);
# electronics_bom_manual.yaml = hand-picked part numbers (no LLM); vague_request.yaml = hard case 9
req = pipeline.load_request(ROOT / "examples" / "electronics_bom.yaml")
packs = steps.load_packs()
cd = Crustdata(use_cache=True)       # False = always hit the API (costs credits)
run = Run("notebook")
run.record("00_request", req)
print(run.dir)
req

## Step 1: understand the specs
Gemini reads the founder's text into specs and candidate part numbers. Nothing it suggests is trusted until
step 2 finds it on a live distributor page. Lines it can't pin down become questions and are held back.

In [ ]:
sr = run.step("01_spec", steps.understand_specs, req, packs)
pack, specs = packs[sr.category], sr.specs
print(f"{sr.parsed_by}: {sr.category}, {sr.units} units -> {sr.deliver_to} by {sr.deadline}, "
      f"data budget ₹{sr.data_budget_inr}")
view.specs(sr)

In [ ]:
# Questions for the founder, and what the model assumed
display(pd.DataFrame([q.model_dump() for q in sr.questions]))
pd.DataFrame({"assumption": sr.assumptions})

To answer, add the answers and re-run the step 1 cell, e.g.
`req.answers = ["Q: voltage rating for the 10µF caps? A: 16V or more, X5R"]`

## Step 2: find suppliers, one part
`PART` is a line_id from the table above.

In [ ]:
PART = specs[0].line_id
spec = next(s for s in specs if s.line_id == PART)
find = run.step("02_find", steps.find_suppliers, spec, pack, cd, part=PART)
print(find.status.value, "| tried:", find.tried, "\n", find.reason)
view.quotes(find)

Price tiers behind each quote (the pipeline picks the cheapest tier we qualify for at our quantity).

In [ ]:
view.tiers(find)

Other parts seen on the way: different part numbers, or the same part number from another maker (possible clones). Substitute candidates later.

In [ ]:
view.quotes(find, alternatives=True)

Unknown sellers from the open search. Step 3 will check these; `domain` is empty when the link is a Google redirect.

In [ ]:
view.leads(find)

The API calls this step made, with credits and cache hits.

In [ ]:
view.calls(run.records[-1:])

### Drill into a raw page
The exact page text a quote was read from.

In [ ]:
q = next((q for q in find.quotes if q.cache_key and q.status.value == "FOUND"), None)
if q:
    print(q.vendor, q.url, "| cache_key:", q.cache_key)
    print(view.page_text(q.cache_key, chars=1500, around="Stock"))

## Steps 2 to 6 for the whole BOM
Step 3 (check), 4 (buy data) and 6 (order) are stubs; step 5 picks the cheapest unflagged quote and sends
cheaper flagged ones (possible clones) to the founder.

In [ ]:
finds = {s.line_id: run.step("02_find", steps.find_suppliers, s, pack, cd, part=s.line_id) for s in specs}
pd.DataFrame([{"part": k, "status": f.status.value, "reason": f.reason} for k, f in finds.items()])

In [ ]:
checks = run.step("03_check", steps.check_suppliers, list(finds.values()))
pd.DataFrame([c.model_dump() for c in checks])

In [ ]:
gaps = run.step("04_buy_data", steps.buy_data, list(finds.values()), sr.data_budget_inr or 0)
pd.DataFrame(gaps)

In [ ]:
decisions = {s.line_id: run.step("05_decide", steps.decide, s, finds[s.line_id], checks, part=s.line_id) for s in specs}
display(pd.DataFrame([{"part": k, "status": d.status, "picked": d.picked.vendor if d.picked else None,
                       "total_inr": d.total_inr, "reason": d.reason} for k, d in decisions.items()]))
# Flagged options (e.g. possible clones) waiting for the founder
pd.DataFrame([{"part": k, **f} for k, d in decisions.items() for f in d.founder_review])

In [ ]:
orders = {k: run.step("06_order", steps.place_order, d, run.id, part=k) for k, d in decisions.items()}
pd.DataFrame([o.model_dump() for o in orders.values() if o])

## This run, step by step

In [ ]:
view.steps(run.records)

## Look at any past run
Includes runs from the CLI (`uv run python -m desk run examples/electronics_bom.yaml`).

In [ ]:
[p.name for p in list_runs()[:10]]

In [ ]:
records = load_run()                 # latest run; or load_run("20261122-131235_electronics_bom")
view.steps(records)

In [ ]:
rec = next(r for r in records if r["step"] == "02_find")
print(rec["part"])
print(json.dumps(rec["output"], indent=1, ensure_ascii=False)[:4000])